# Session 3 — Testing Hypotheses

**Where we are.** Session 2 gave us a cleaned file and three untested patterns. Today, for each pattern: could it be chance, how big is it, and what does it mean?

We start from **your** cleaned file, not from the raw data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

df = pd.read_csv("data/clean/clients_clean.csv")

# a CSV does not store category types: set them again
order = ["graduate school", "university", "high school", "others"]
df["education"] = pd.Categorical(df["education"], categories=order, ordered=True)
df.shape

## Part 1 — What a test does

$H_0$ for RQ2: defaulters and non-defaulters have the same mean credit limit.

If $H_0$ were true, the default label would be unrelated to the limit. So we can **shuffle** the labels and see what gaps chance alone produces.

In [ ]:
rng = np.random.default_rng(1)
limit, default = df["limit"].to_numpy(), df["default"].to_numpy()

observed = limit[default == 1].mean() - limit[default == 0].mean()

shuffled_gaps = []
for _ in range(1000):
    s = rng.permutation(default)
    shuffled_gaps.append(limit[s == 1].mean() - limit[s == 0].mean())

print(f"observed gap: {observed:,.0f}")
print(f"largest shuffled gap: {np.max(np.abs(shuffled_gaps)):,.0f}")

In [ ]:
plt.hist(shuffled_gaps, bins=40, color="grey")
plt.axvline(observed, color="orangered")
plt.title("Chance alone never produces a gap this large")
plt.xlabel("Gap in mean limit (NT$)")
plt.show()

## Part 2 — Comparing two groups (RQ2)

### Welch's t-test

In [ ]:
yes = df.loc[df["default"] == 1, "limit"]
no = df.loc[df["default"] == 0, "limit"]

res = stats.ttest_ind(yes, no, equal_var=False)
print(res.statistic, res.pvalue)
print(res.confidence_interval())

### Effect size: Cohen's d

In [ ]:
pooled_sd = np.sqrt((yes.var() + no.var()) / 2)
d = (yes.mean() - no.mean()) / pooled_sd
round(d, 2)

### A test that does not assume normality

In [ ]:
stats.mannwhitneyu(yes, no).pvalue

### Significance depends on sample size
The same gap, tested on random samples of different sizes.

In [ ]:
for n in [50, 100, 200, 400, 800]:
    sig = 0
    for _ in range(200):
        s = df.sample(n)
        p = stats.ttest_ind(s.loc[s["default"] == 1, "limit"],
                            s.loc[s["default"] == 0, "limit"], equal_var=False).pvalue
        sig += p < 0.05
    print(f"n = {n:4d}: significant in {sig / 200:.0%} of samples")

## Part 3 — Categorical data (RQ1)

### Chi-square test

In [ ]:
table = pd.crosstab(df["education"], df["default"])
res = stats.chi2_contingency(table)
print(res.statistic, res.pvalue, res.dof)
print(pd.DataFrame(res.expected_freq, index=table.index, columns=table.columns).round(0))

### Effect size: Cramér's V

In [ ]:
n = table.values.sum()
v = np.sqrt(res.statistic / (n * (min(table.shape) - 1)))
round(v, 3)

### The black lines from Session 2: 95% confidence intervals

In [ ]:
ax = sns.pointplot(data=df, x="education", y="default", linestyle="none", capsize=0.2)
ax.set(title="Default rate by education, with 95% confidence intervals",
       xlabel="", ylabel="Default rate")
plt.show()

## Part 4 — Linear regression (RQ3)

### One variable: the Session 2 chart as a line

In [ ]:
simple = smf.ols("default ~ late_months", data=df).fit(cov_type="HC1")
simple.params

### Several variables at once

In [ ]:
df["limit10k"] = df["limit"] / 10_000
full = smf.ols("default ~ late_months + limit10k + age + C(sex) + C(education)",
               data=df).fit(cov_type="HC1")
full.summary()

### How the limit coefficient changes when we add controls

In [ ]:
formulas = {
    "limit only": "default ~ limit10k",
    "+ late months": "default ~ limit10k + late_months",
    "+ age, sex, education": "default ~ limit10k + late_months + age + C(sex) + C(education)",
}
for name, f in formulas.items():
    m = smf.ols(f, data=df).fit(cov_type="HC1")
    print(f"{name:25s} {m.params['limit10k']:.4f}   R2 = {m.rsquared:.3f}")

### Check the predictions

In [ ]:
pred = full.fittedvalues
print((pred < 0).sum(), pred.min().round(3), pred.max().round(3))

Optional: a **logit** model keeps predicted probabilities between 0 and 1.

In [ ]:
logit = smf.logit("default ~ late_months + limit10k + age + C(sex) + C(education)",
                  data=df).fit(disp=False)
logit.predict(df).describe().round(3)

---
## Part 5 — In-class exercise (feeds your group presentation)

Use **your group's** cleaned data and **your** research question.

### Task 1 — State $H_0$ and $H_1$ for your research question

*Your answer:*

### Task 2 — Run the test that fits your question

In [ ]:
# TODO: your code here


### Task 3 — Report the p-value, an effect size and a confidence interval
Collect them in a small table (a DataFrame or a markdown table).

In [ ]:
# TODO: your code here


### Task 4 — Three sentences
What the result means, how big it is, and one limit.

*Your answer:*

### Task 5 — AI disclosure
If you used an AI tool, state which one and for which cells. Then state how you checked its output.

*Your answer:*